# BrainScan-v2 — Étape 1 : Classification des tumeurs cérébrales (GoogLeNet)

## 1.1 — Exploration du dataset (EDA)

**Objectifs du notebook**

1. Vérifier l'intégrité du dataset : extensions autorisées (`jpeg`, `jpg`, `bmp`, `png`) et
   fichiers illisibles (chargements protégés par `try/except`).
2. Découvrir les classes à partir des **noms de dossiers**.
3. Représenter le nombre d'échantillons de chaque classe à l'aide d'un **countplot**.
4. Afficher un **échantillon d'images pour chaque classe**.
5. Relever les statistiques utiles à l'étape de préparation (dimensions, modes,
   déséquilibre entre classes).

> **Jeu de données** : IRM cérébrales issues de la base *Brain Tumor MRI Dataset*
> (classification de tumeurs). Les classes correspondent aux sous-dossiers de
> `data/Data/` : `glioma`, `meningioma`, `notumor`, `pituitary`.
>
> **Durée d'exécution** : ~1 minute (scan du système de fichiers + ouverture des images).


In [ ]:
# Cellule 0 — Imports, configuration et chemins du projet
"""Cellules communes réutilisées par les 4 notebooks de l'étape 1 (classification)."""

import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib

if matplotlib.get_backend().lower() == "agg":  # exécution headless (Docker / nbconvert)
    try:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    except Exception:
        pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# --------------------------------------------------------------------------------------
# Configuration du projet
# --------------------------------------------------------------------------------------
def _detect_project_root() -> Path:
    """Racine du projet (compatible exécution locale et exécution dans Docker /app)."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "Data").is_dir() and (candidate / "src").is_dir():
            return candidate
    if Path("/app/data/Data").is_dir():
        return Path("/app")
    raise FileNotFoundError("Dataset introuvable : « data/Data » manquant depuis " + str(Path.cwd()))


PROJECT_ROOT = _detect_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed" / "classification"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for _d in (ARTIFACTS_DIR, MODELS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

#: Graine aléatoire globale (reproductibilité de bout en bout).
SEED = 42

#: Extensions d'images autorisées par le cahier des charges.
ALLOWED_EXTENSIONS = {".jpeg", ".jpg", ".bmp", ".png"}

#: Libellés lisibles pour l'affichage (correspondance dossier -> pathologie).
CLASS_DISPLAY = {
    "glioma": "Gliome",
    "meningioma": "Méningiome",
    "notumor": "Aucune tumeur",
    "pituitary": "Hypophyse",
}

#: Classes du dataset, déduites des noms de dossiers (ordre alphabétique).
class_names = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_DIR         : {RAW_DIR}")
print(f"PROCESSED_DIR   : {PROCESSED_DIR}")
print(f"ARTIFACTS_DIR   : {ARTIFACTS_DIR}")
print(f"MODELS_DIR      : {MODELS_DIR}")
print(f"Classes         : {class_names}")


### 1.2 — Environnement d'exécution

Vérification des versions des bibliothèques utilisées (TensorFlow / Keras / Pillow / Seaborn)
et des périphériques de calcul disponibles.

In [ ]:
import sys as _sys

import keras
import tensorflow as tf
from PIL import Image

print(f"Python      : {_sys.version.split()[0]}")
print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"Pillow      : {Image.__version__ if hasattr(Image, '__version__') else 'n/a'}")
print(f"Seaborn     : {sns.__version__}")
print(f"Devices     : {tf.config.list_physical_devices() or 'CPU uniquement'}")


### 1.3 — Contrôle des extensions de fichiers

Les dossiers de classes sont parcourus récursivement. Chaque fichier est classé selon son
extension ; **les fichiers dont l'extension n'est pas dans la liste autorisée sont écartés**
avant tout traitement ultérieur.

In [ ]:
# Cellule 1.3 — Inventaire des fichiers et filtrage par extension
rows = []
for class_dir in sorted(p for p in RAW_DIR.iterdir() if p.is_dir()):
    for f in sorted(class_dir.rglob("*")):
        if not f.is_file():
            continue
        ext = f.suffix.lower()
        rows.append(
            {
                "label": class_dir.name,
                "file": f.name,
                "path": str(f),
                "extension": ext,
                "allowed": ext in ALLOWED_EXTENSIONS,
            }
        )

files_df = pd.DataFrame(rows)

print(f"Fichiers découverts sous {RAW_DIR} : {len(files_df)}")
print(f"Extensions autorisées              : {sorted(ALLOWED_EXTENSIONS)}")

files_df.groupby(["label", "extension"]).size().rename("nb_fichiers").reset_index()


In [ ]:
# Cellule 1.4 — Fichiers écartés (extension non autorisée)
rejected_df = files_df.loc[~files_df["allowed"]].copy()

print(f"Nombre de fichiers écartés : {len(rejected_df)}")
if len(rejected_df):
    rejected_df[["label", "file", "extension"]].head(50)
else:
    print("Aucun : toutes les extensions présentes sont autorisées.")


### 1.4 — Chargement des images et gestion des erreurs (`try/except`)

Chaque image candidate est ouverte avec Pillow puis **vérifiée** (`Image.verify()`).
Toute exception (fichier corrompu, image tronquée, droits insuffisants…) est capturée,
journalisée dans un tableau `failures`, et l'image est écartée sans interrompre le
traitement.

In [ ]:
# Cellule 1.5 — Chargement sécurisé (try/except) + métadonnées
manifest_rows: list[dict] = []
failures: list[dict] = []

for rec in files_df.loc[files_df["allowed"]].to_dict("records"):
    try:
        with Image.open(rec["path"]) as im:
            im.verify()  # contrôle d'intégrité de la structure du fichier
        with Image.open(rec["path"]) as im:
            width, height, mode, fmt = im.width, im.height, im.mode, im.format
        manifest_rows.append(
            {
                **rec,
                "width": width,
                "height": height,
                "mode": mode,
                "format": fmt,
            }
        )
    except Exception as exc:  # noqa: BLE001 - on veut capturer toute erreur de lecture
        failures.append({"path": rec["path"], "error": f"{type(exc).__name__}: {exc}"})

manifest_df = pd.DataFrame(manifest_rows)
failures_df = pd.DataFrame(failures)

print(f"Images chargées avec succès : {len(manifest_df)}")
print(f"Images écartées (erreur)     : {len(failures_df)}")
if len(failures_df):
    failures_df.head(20)


### 1.5 — Découverte des classes

Les classes sont déduites des **noms de dossiers**. Leur ordre alphabétique définit
l'ordre d'encodage utilisé par `tf.keras.utils.image_dataset_from_directory()`.

In [ ]:
# Cellule 1.6 — Classes du dataset
class_names = sorted(manifest_df["label"].unique().tolist())

print(f"Nombre de classes détectées : {len(class_names)}\n")
for i, name in enumerate(class_names):
    print(f"  [{i}] {name:<12s} -> {CLASS_DISPLAY.get(name, name)}")

print("\nCet ordre (alphabétique) sera celui des labels encodés par image_dataset_from_directory().")


### 1.6 — Répartition des échantillons (`countplot`)

Représentation du nombre d'échantillons par classe, avec le pourcentage et le **taux de
déséquilibre** (max/min) qui guidera l'augmentation des données à l'étape 2.

In [ ]:
# Cellule 1.7 — Statistiques par classe
counts_df = (
    manifest_df["label"]
    .value_counts()
    .reindex(class_names)
    .rename_axis("label")
    .reset_index(name="nb_images")
)
counts_df["classe"] = counts_df["label"].map(lambda c: CLASS_DISPLAY.get(c, c))
counts_df["pct"] = (counts_df["nb_images"] / counts_df["nb_images"].sum() * 100).round(2)
counts_df


In [ ]:
# Cellule 1.8 — Countplot du nombre d'échantillons par classe
fig, ax = plt.subplots(figsize=(9, 5))
sns.countplot(
    data=manifest_df,
    x="label",
    order=class_names,
    hue="label",
    palette="viridis",
    legend=False,
    ax=ax,
)
for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=2)

ax.set_title("Nombre d'images par classe — dataset de classification", fontsize=13, weight="bold")
ax.set_xlabel("Classe (dossier source)")
ax.set_ylabel("Nombre d'images")
ax.set_xticklabels([CLASS_DISPLAY.get(c, c) for c in class_names], rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Cellule 1.9 — Taux de déséquilibre entre classes
imbalance = counts_df["nb_images"].max() / counts_df["nb_images"].min()
print(f"Classe la plus représentée : {counts_df.loc[counts_df['nb_images'].idxmax(), 'classe']} "
      f"({counts_df['nb_images'].max()} images)")
print(f"Classe la moins représentée: {counts_df.loc[counts_df['nb_images'].idxmin(), 'classe']} "
      f"({counts_df['nb_images'].min()} images)")
print(f"Taux de déséquilibre (max/min) : {imbalance:.2f}")
print("\n-> Le déséquilibre reste modéré : l'augmentation des données (étape 2) suffira")
print("   à le compenser, sans recours à un ré-échantillonnage pondéré.")


### 1.7 — Échantillon d'images par classe

Aperçu visuel d'images tirées aléatoirement dans chaque classe (graine fixe → résultat
reproductible).

In [ ]:
# Cellule 1.10 — Affichage d'un échantillon d'images pour chaque classe
SAMPLES_PER_CLASS = 6
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(
    len(class_names),
    SAMPLES_PER_CLASS,
    figsize=(2.0 * SAMPLES_PER_CLASS, 2.2 * len(class_names)),
)

for r, label in enumerate(class_names):
    paths = manifest_df.loc[manifest_df["label"] == label, "path"].to_numpy()
    picked = rng.choice(paths, size=min(SAMPLES_PER_CLASS, len(paths)), replace=False)
    for c, path in enumerate(picked):
        ax = axes[r, c]
        with Image.open(path) as im:
            ax.imshow(im.convert("RGB"))
        ax.set_title(Path(path).name[:24], fontsize=7)
        ax.axis("off")
    axes[r, 0].set_ylabel(CLASS_DISPLAY.get(label, label), fontsize=12, weight="bold")

fig.suptitle("Échantillon d'images par classe", fontsize=14, weight="bold", y=1.001)
plt.tight_layout()
plt.show()


### 1.8 — Statistiques descriptives

Dimensions, modes de couleur et répartition des tailles : ces informations déterminent
le prétraitement à appliquer (conversion en RGB, redimensionnement).

In [ ]:
# Cellule 1.11 — Statistiques par classe (dimensions et modes)
stats_df = (
    manifest_df.groupby("label")
    .agg(
        nb_images=("file", "size"),
        largeur_moy=("width", "mean"),
        hauteur_moy=("height", "mean"),
        largeur_med=("width", "median"),
        hauteur_med=("height", "median"),
        modes=("mode", lambda s: ", ".join(sorted(s.unique()))),
    )
    .reindex(class_names)
    .round(1)
)
stats_df.insert(0, "classe", [CLASS_DISPLAY.get(c, c) for c in stats_df.index])
stats_df


In [ ]:
# Cellule 1.12 — Répartition des modes de couleur et des tailles
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

sns.countplot(data=manifest_df, x="mode", hue="mode", palette="magma", legend=False, ax=ax1)
ax1.set_title("Modes de couleur des images sources", weight="bold")
ax1.set_xlabel("Mode PIL")
ax1.set_ylabel("Nombre d'images")

sizes = manifest_df["width"].astype(str) + "x" + manifest_df["height"].astype(str)
sns.countplot(data=pd.DataFrame({"taille": sizes}), x="taille", hue="taille",
              palette="viridis", legend=False, ax=ax2)
ax2.set_title(f"Tailles les plus fréquentes ({manifest_df['width'].nunique()} distinctes)",
              weight="bold")
ax2.set_xlabel("Dimensions (largeur x hauteur)")
ax2.set_ylabel("Nombre d'images")
ax2.tick_params(axis="x", rotation=90, labelsize=8)

plt.tight_layout()
plt.show()


### 1.9 — Synthèse et sauvegarde du manifeste

Le manifeste (une ligne par image valide) est sauvegardé : il sert de source de vérité
pour l'étape 2 (split) et garantit que les trois ensembles restent strictement disjoints.

In [ ]:
# Cellule 1.13 — Synthèse
n_images = len(manifest_df)
n_classes = len(class_names)
n_sizes = manifest_df[["width", "height"]].drop_duplicates().shape[0]
gray_ratio = manifest_df["mode"].isin(["L", "1", "I", "F"]).mean()

print("=" * 68)
print("SYNTHÈSE — CLASSIFICATION DES TUMEURS (étape 1)")
print("=" * 68)
print(f"Images valides                : {n_images}")
print(f"Images écartées (extension)   : {len(rejected_df)}")
print(f"Images écartées (erreur)       : {len(failures_df)}")
print(f"Classes                       : {n_classes} -> {', '.join(class_names)}")
print(f"Tailles distinctes             : {n_sizes} (redimensionnement obligatoire)")
print(f"Part d'images en niveaux de gris: {gray_ratio:.1%} (conversion RGB requise)")
print(f"Taux de déséquilibre (max/min): {imbalance:.2f}")
print("=" * 68)


In [ ]:
# Cellule 1.14 — Sauvegarde du manifeste (réutilisé par le notebook 2)
MANIFEST_CSV = REPORTS_DIR / "classification_manifest.csv"
manifest_df.to_csv(MANIFEST_CSV, index=False)

eda_summary = {
    "n_images_valid": int(n_images),
    "n_rejected_extension": int(len(rejected_df)),
    "n_failed_load": int(len(failures_df)),
    "classes": class_names,
    "counts": {r["label"]: int(r["nb_images"]) for _, r in counts_df.iterrows()},
    "imbalance_ratio": round(float(imbalance), 3),
    "distinct_sizes": int(n_sizes),
    "grayscale_ratio": round(float(gray_ratio), 4),
    "modes": {k: int(v) for k, v in manifest_df["mode"].value_counts().items()},
}
(REPORTS_DIR / "eda_classification.json").write_text(
    json.dumps(eda_summary, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(f"Manifeste  : {MANIFEST_CSV}")
print(f"Rapport    : {REPORTS_DIR / 'eda_classification.json'}")
